# NB7 · Detector validation sheet

Builds `detector_validation.xlsx` — the hand-labelling workbook for the construct-validity
check. It draws a stratified sample of workflow files, writes each one out in full next to
the question the label answers, and saves the classifier's own verdicts to a **separate**
key file so the labelling is blind.

**Sample design** (175 rows)

| stratum | n | what it measures |
|---|---|---|
| predicted positive, 20 per class | 100 | precision |
| **near-miss** negatives, 12 per class | 60 | recall, sampled adversarially |
| uniform random negatives | 15 | sanity check |

Near-misses are files whose raw text matches a safeguard's keyword signature but which the
classifier still rejected — almost always because it found no `pull_request` trigger. That
is where false negatives actually live, so sampling there bounds recall from below far more
tightly than uniform sampling, which would need thousands of rows to find a single miss.

Only the five *workflow* detectors are validated here. `C1_codeowners`, `T5_depbot` and
`T7_precommit` are file-existence checks over repository configs, not workflow classifiers,
and need no YAML judgement.

**Run cells 1–7 now** (~15 minutes), label the workbook by hand, then come back and run
cell 8 to score it.

In [ ]:
#@title Configuration { display-mode: "form" }
N_POS_PER_CLASS  = 20   #@param {type:"integer"}
N_NEAR_PER_CLASS = 12   #@param {type:"integer"}
N_RANDOM         = 15   #@param {type:"integer"}
SUBSAMPLE        = 8    #@param {type:"integer"}
MAX_CHARS        = 4000 #@param {type:"integer"}
SEED             = 20261005 #@param {type:"integer"}

import os, sys, json, re, time, hashlib, tarfile, gzip, csv, random
from pathlib import Path
IN_COLAB=False
try:
    from google.colab import drive; IN_COLAB=True
except ImportError: pass
DRIVE_OK=False
if IN_COLAB:
    try: drive.mount("/content/drive",force_remount=False); DRIVE_OK=True
    except Exception as e: print("!! Drive mount FAILED:",e)
DRIVE=("/content/drive/MyDrive/peltzman" if DRIVE_OK else
       "/content/peltzman_LOCAL" if IN_COLAB else os.path.expanduser("~/peltzman_out"))
STATE=Path(DRIVE,"state"); RAW=Path(DRIVE,"raw"); OUT=Path(DRIVE,"results")
CODE=Path("/content" if IN_COLAB else "/tmp")
for _p in (STATE,RAW,OUT,CODE): Path(_p).mkdir(parents=True,exist_ok=True)
sys.path.insert(0,str(CODE))
print("state:",STATE,"\nraw  :",RAW)

In [ ]:
%%writefile /content/peltzman_io.py
"""
Peltzman Protocol - durable, resumable job state.

Guarantee
---------
Kill the process, lose the runtime, close the browser, at ANY instant:
  * no file is ever left half-written,
  * no completed work is ever lost,
  * no work is ever silently duplicated,
  * restarting the cell resumes from the last completed batch.

How
---
The key list is fixed and ordered at job creation, and batch `i` is defined as
a deterministic slice of it. A batch's output is therefore a pure function of
its index, so re-running a batch overwrites its own file and can never
duplicate rows. Completion is represented by the *existence* of that file and
nothing else - there is no separate done-list that can drift out of sync with
the data.

Every file is written to a temporary name, flushed, fsync'd, and then moved
into place with os.replace(), which is atomic. A crash mid-write leaves a
stray .tmp file (harmless, cleaned on next run) and the previous state intact.

Because completion is file existence, the crash window is exactly one batch.
Size batches so one takes a few minutes.
"""
from __future__ import annotations
import json, os, time, hashlib, tempfile, shutil, glob, sys
from pathlib import Path

__all__ = ["Job", "atomic_write_text", "atomic_write_bytes"]


# --------------------------------------------------------------- atomic io --
def atomic_write_bytes(path, data: bytes):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=str(path.parent), prefix=path.name + ".", suffix=".tmp")
    try:
        with os.fdopen(fd, "wb") as f:
            f.write(data)
            f.flush()
            os.fsync(f.fileno())          # no-op on some mounts; harmless
        os.replace(tmp, path)             # atomic within a filesystem
    except BaseException:
        try: os.unlink(tmp)
        except OSError: pass
        raise


def atomic_write_text(path, text: str):
    atomic_write_bytes(path, text.encode("utf-8"))


def _hash_params(params) -> str:
    blob = json.dumps(params, sort_keys=True, default=str).encode()
    return hashlib.sha1(blob).hexdigest()[:10]


# --------------------------------------------------------------------- job --
class Job:
    """A resumable pass over an ordered list of keys.

    >>> job = Job("nb1_workflows", keys=repos, root=DRIVE, batch_size=25,
    ...           params={"detector": "v3"})
    >>> for i, keys in job.pending():
    ...     job.commit(i, [row for k in keys for row in scan(k)])
    >>> df = job.frame()

    Changing `params` starts a new, separate run directory: results computed
    under different settings are never silently mixed.
    """

    def __init__(self, name, keys, root, batch_size=25, params=None, verbose=True):
        self.name = name
        self.params = dict(params or {})
        self.batch_size = int(batch_size)
        self.verbose = verbose
        # NOTE: the key count is deliberately NOT part of the run identity.
        # If the input list changes at all we want a loud error, not a silent
        # fresh run that looks like "nothing was ever done".
        self.dir = Path(root) / name / _hash_params(
            {"params": self.params, "batch_size": self.batch_size})
        self.shards = self.dir / "shards"
        self.shards.mkdir(parents=True, exist_ok=True)

        keyfile = self.dir / "keys.json"
        if keyfile.exists():
            stored = json.loads(keyfile.read_text())
            if stored != list(keys):
                raise ValueError(
                    f"{self.name}: key list changed since this run started "
                    f"({len(stored)} -> {len(keys)}). Change `params` to start "
                    f"a new run, or restore the original list.")
        else:
            atomic_write_text(keyfile, json.dumps(list(keys)))
            atomic_write_text(self.dir / "meta.json", json.dumps({
                "name": name, "created": time.time(), "n_keys": len(keys),
                "batch_size": self.batch_size, "params": self.params,
                "python": sys.version.split()[0]}, indent=1))

        self.keys = list(keys)
        self._sweep_tmp()
        if verbose:
            d, n = self.n_done_batches(), self.n_batches()
            print(f"[{name}] {self.dir}")
            print(f"[{name}] {d}/{n} batches complete"
                  + (f" — resuming at batch {d}" if 0 < d < n else
                     " — nothing to do" if d == n else " — starting fresh"))

    # ---------------------------------------------------------- internals --
    def _sweep_tmp(self):
        """Remove leftovers from a crash mid-write. They are never valid."""
        for t in glob.glob(str(self.shards / "*.tmp")) + glob.glob(str(self.dir / "*.tmp")):
            try: os.unlink(t)
            except OSError: pass

    def _path(self, i) -> Path:
        return self.shards / f"batch-{i:06d}.jsonl"

    def n_batches(self) -> int:
        return (len(self.keys) + self.batch_size - 1) // self.batch_size

    def batch_keys(self, i):
        return self.keys[i * self.batch_size:(i + 1) * self.batch_size]

    def done_batches(self):
        return {i for i in range(self.n_batches()) if self._path(i).exists()}

    def n_done_batches(self) -> int:
        return len(self.done_batches())

    # ------------------------------------------------------------- driving --
    def pending(self):
        """Yield (batch_index, keys) for every batch not yet committed."""
        done = self.done_batches()
        for i in range(self.n_batches()):
            if i not in done:
                yield i, self.batch_keys(i)

    def commit(self, i, rows):
        """Atomically record this batch's output. Idempotent by construction."""
        payload = "".join(json.dumps(r, separators=(",", ":"), default=str) + "\n"
                          for r in rows)
        atomic_write_text(self._path(i), payload)
        self.beat(i, len(rows))

    def beat(self, i, n_rows=None, note=None):
        atomic_write_text(self.dir / "heartbeat.json", json.dumps({
            "at": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
            "last_batch": i, "rows": n_rows, "note": note,
            "done": self.n_done_batches(), "total": self.n_batches()}, indent=1))

    def run(self, fn, on_error="record", progress_every=1, after_batch=None):
        """Drive the whole job. `fn(key) -> list[dict]`.

        on_error='record'  capture the failure as a row and move on (default)
                 'skip'    drop the key silently
                 'raise'   stop, keeping every committed batch
        after_batch(job, i)  optional hook run after each successful commit,
                 e.g. to replicate state off the runtime. A failing hook never
                 loses the batch - it is already on disk before the hook runs.
        """
        t0, n_before = time.time(), self.n_done_batches()
        for i, keys in self.pending():
            rows = []
            for k in keys:
                try:
                    rows.extend(fn(k) or [])
                except KeyboardInterrupt:
                    raise
                except Exception as e:
                    if on_error == "raise":
                        raise
                    if on_error == "record":
                        rows.append({"_key": k, "_error": f"{type(e).__name__}: {e}"})
            self.commit(i, rows)
            if after_batch is not None:
                try: after_batch(self, i)
                except Exception as e:
                    print(f"  after_batch hook failed ({type(e).__name__}: {e}) — "
                          f"batch is still committed locally")
            done = self.n_done_batches()
            if self.verbose and (done - n_before) % progress_every == 0:
                el = time.time() - t0
                per = el / max(1, done - n_before)
                left = (self.n_batches() - done) * per
                print(f"  batch {done}/{self.n_batches()}  "
                      f"{per:5.1f}s/batch  ~{left/60:5.1f} min remaining")
        if self.verbose:
            print(f"[{self.name}] complete: {self.n_batches()} batches")

    # ------------------------------------------------------------ reading --
    def rows(self, drop_errors=True):
        """Iterate every committed row. Truncated trailing lines - which an
        atomic write cannot produce, but a copied/synced file might - are
        skipped and counted rather than raising."""
        bad = 0
        for i in sorted(self.done_batches()):
            with open(self._path(i), "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        r = json.loads(line)
                    except json.JSONDecodeError:
                        bad += 1
                        continue
                    if drop_errors and isinstance(r, dict) and "_error" in r:
                        continue
                    yield r
        if bad and self.verbose:
            print(f"[{self.name}] warning: skipped {bad} unparseable line(s) — "
                  f"run .verify() to locate")

    def frame(self, drop_errors=True):
        import pandas as pd
        return pd.DataFrame(list(self.rows(drop_errors=drop_errors)))

    def errors(self):
        return [r for r in self.rows(drop_errors=False)
                if isinstance(r, dict) and "_error" in r]

    # -------------------------------------------------------- maintenance --
    def verify(self):
        """Check every committed shard parses. Returns list of bad batch ids."""
        bad = []
        for i in sorted(self.done_batches()):
            try:
                for line in open(self._path(i), encoding="utf-8"):
                    if line.strip():
                        json.loads(line)
            except Exception as e:
                bad.append((i, f"{type(e).__name__}: {e}"))
        if self.verbose:
            print(f"[{self.name}] verify: {self.n_done_batches()} shards, "
                  f"{len(bad)} damaged")
        return bad

    def reset(self, batches=None):
        """Delete shards so they are recomputed. `batches=None` resets damaged
        ones only; pass an iterable of indices, or 'all'."""
        if batches == "all":
            targets = sorted(self.done_batches())
        elif batches is None:
            targets = [i for i, _ in self.verify()]
        else:
            targets = list(batches)
        for i in targets:
            try: os.unlink(self._path(i))
            except OSError: pass
        if self.verbose:
            print(f"[{self.name}] reset {len(targets)} batch(es)")
        return targets

    def retry_errors(self):
        """Re-queue every batch that recorded at least one failure."""
        hit = set()
        for i in sorted(self.done_batches()):
            for line in open(self._path(i), encoding="utf-8"):
                if '"_error"' in line:
                    hit.add(i); break
        return self.reset(hit)

    def export(self, path, fmt="parquet"):
        """Write the whole result set as one file, atomically."""
        df = self.frame()
        p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
        tmp = p.with_suffix(p.suffix + ".tmp")
        if fmt == "parquet":
            df.to_parquet(tmp, index=False)
        else:
            df.to_csv(tmp, index=False)
        os.replace(tmp, p)
        if self.verbose:
            print(f"[{self.name}] exported {len(df):,} rows -> {p}")
        return p

    def status(self):
        hb = self.dir / "heartbeat.json"
        return {"dir": str(self.dir), "batches": self.n_batches(),
                "done": self.n_done_batches(), "keys": len(self.keys),
                "heartbeat": json.loads(hb.read_text()) if hb.exists() else None}


In [ ]:
%%writefile /content/detectors.py
"""
Peltzman Protocol - safeguard detectors.
Classifies a GitHub Actions workflow file (raw YAML text) into safeguard
categories, and computes cohort/bundling statistics from adoption dates.

Design notes
------------
* Detection is two-stage: a cheap regex prefilter over raw text, then a YAML
  parse only for files that matched something. On ~10M historical file
  versions the prefilter is what makes NB1 finish in hours instead of days.
* The YAML `on:` key is parsed by PyYAML (YAML 1.1) as the BOOLEAN True, not
  the string "on". Every workflow parser that forgets this silently finds no
  triggers. handled in `_get_triggers`.
"""
import re
from collections import defaultdict

# --- safeguard signatures -------------------------------------------------
# Ordered dict of treatment id -> list of compiled patterns over raw file text.
_RAW = {
    "T1_test": [
        r"\bpytest\b", r"\bvitest\b", r"\bjest\b", r"\bgo\s+test\b",
        r"\bmvn\b[^\n]*\btest\b", r"\bgradle\w*\b[^\n]*\btest\b",
        r"\bcargo\s+test\b", r"\brspec\b", r"\bnpm\s+(?:run\s+)?test\b",
        r"\byarn\s+test\b", r"\btox\b", r"\bdotnet\s+test\b",
        r"\bphpunit\b", r"\bctest\b", r"\bpython\s+-m\s+unittest\b",
        r"\bbundle\s+exec\s+rspec\b",
    ],
    "T2_lint": [
        r"\beslint\b", r"\bruff\b", r"\bflake8\b", r"\bgolangci-lint\b",
        r"\bcheckstyle\b", r"\bspotbugs\b", r"\bcargo\s+clippy\b",
        r"\brubocop\b", r"\bpylint\b", r"\bshellcheck\b", r"\bbiome\s+lint\b",
    ],
    "T3_coverage": [
        r"codecov/codecov-action", r"coverallsapp/", r"\bjacoco\b",
        r"--cov-fail-under", r"--fail-under", r"\bcoveralls\b",
        r"\bcodecov\b",
    ],
    "T4_types": [
        r"\bmypy\b", r"\bpyright\b", r"(?<![\w-])tsc(?![\w-])",
        r"\bpsalm\b", r"\bphpstan\b", r"\bflow\s+check\b",
    ],
    "T6_mergequeue": [
        r"\bmerge_group\b",
    ],
}
PATTERNS = {k: [re.compile(p, re.I) for p in v] for k, v in _RAW.items()}
TREATMENTS = list(PATTERNS.keys())

# YAML loading. PyYAML's pure-Python SafeLoader runs ~1.8 ms on a typical
# workflow; the libyaml-backed CSafeLoader does the same file in ~0.19 ms.
# Across millions of files that is the difference between two hours and twenty
# minutes, so prefer it and fall back only if the C extension is absent.
import yaml
try:
    _LOADER = yaml.CSafeLoader
    YAML_BACKEND = "libyaml (CSafeLoader)"
except AttributeError:                       # PyYAML built without libyaml
    _LOADER = yaml.SafeLoader
    YAML_BACKEND = "pure python (SafeLoader)"

def _yaml_load(text):
    return yaml.load(text, Loader=_LOADER)


# Files that are workflow definitions but never act as a PR gate.
_NON_GATE_HINT = re.compile(r"\b(stale|greetings|labeler|first-interaction)\b", re.I)


def prefilter(text):
    """Cheap pass over raw text. Returns the set of treatments whose signature
    appears anywhere in the file. No YAML parsing."""
    hits = set()
    for t, pats in PATTERNS.items():
        for p in pats:
            if p.search(text):
                hits.add(t)
                break
    return hits


def _get_triggers(doc):
    """Return the set of trigger names for a parsed workflow document.

    PyYAML resolves the bare key `on` to the boolean True (YAML 1.1). Look
    under both spellings or you will find no triggers at all.
    """
    if not isinstance(doc, dict):
        return set()
    node = doc.get("on", doc.get(True))
    if node is None:
        return set()
    if isinstance(node, str):
        return {node}
    if isinstance(node, list):
        return {str(x) for x in node}
    if isinstance(node, dict):
        return {str(k) for k in node.keys()}
    return set()


def classify(text, require_pr_trigger=True):
    """Full classification of one workflow file version.

    Returns (treatments:set, meta:dict). `meta` carries the triggers found and
    why a file was rejected, so the prototype can be audited by hand.
    """
    meta = {"triggers": set(), "parsed": False, "reason": None}
    hits = prefilter(text)
    if not hits:
        meta["reason"] = "no signature"
        return set(), meta

    try:
        doc = _yaml_load(text)
        meta["parsed"] = True
    except Exception as e:                       # malformed historical version
        meta["reason"] = "unparseable: %s" % type(e).__name__
        return set(), meta

    triggers = _get_triggers(doc)
    meta["triggers"] = triggers

    # merge_group is a trigger, not a job step: judge it on triggers alone.
    out = set()
    if "merge_group" in triggers:
        out.add("T6_mergequeue")

    gate_hits = hits - {"T6_mergequeue"}
    if gate_hits:
        if require_pr_trigger and "pull_request" not in triggers \
                and "pull_request_target" not in triggers:
            meta["reason"] = "no pull_request trigger"
        elif _NON_GATE_HINT.search(str(doc.get("name", ""))):
            meta["reason"] = "housekeeping workflow"
        else:
            out |= gate_hits
    return out, meta


# --- cohort / bundling ----------------------------------------------------

def _m(ym):
    """'YYYY-MM' -> integer month index, for arithmetic."""
    y, mth = ym.split("-")
    return int(y) * 12 + int(mth) - 1


def first_adoption(observations):
    """observations: iterable of (repo, treatment, 'YYYY-MM').
    Returns {repo: {treatment: earliest 'YYYY-MM'}}."""
    out = defaultdict(dict)
    for repo, t, ym in observations:
        cur = out[repo].get(t)
        if cur is None or _m(ym) < _m(cur):
            out[repo][t] = ym
    return dict(out)


def bundling_rate(adoptions, window_months=3):
    """Share of repos that adopt two or more DISTINCT safeguards within
    +/- `window_months` of each other. This is the day-1 kill criterion: above
    0.50 the per-safeguard design cannot separate treatments.

    Returns (rate, n_repos_with_2plus, n_repos_bundled).
    """
    bundled = multi = 0
    for repo, d in adoptions.items():
        months = sorted(_m(v) for v in d.values())
        if len(months) < 2:
            continue
        multi += 1
        if any(months[i + 1] - months[i] <= window_months
               for i in range(len(months) - 1)):
            bundled += 1
    rate = (bundled / multi) if multi else 0.0
    return rate, multi, bundled


def cohort_table(adoptions, treatment):
    """Adopters per calendar month for one treatment -> {'YYYY-MM': count}."""
    counts = defaultdict(int)
    for repo, d in adoptions.items():
        if treatment in d:
            counts[d[treatment]] += 1
    return dict(sorted(counts.items()))


In [ ]:
import importlib, peltzman_io, detectors
for m in (peltzman_io, detectors): importlib.reload(m)
from peltzman_io import Job, atomic_write_text
from detectors import PATTERNS, TREATMENTS, classify, prefilter

N_FILES = json.loads(Path(STATE,"nb1_member_count.json").read_text())["n"]
CHUNK   = 50_000
N_CHUNKS = (N_FILES + CHUNK - 1)//CHUNK
cls = Job("nb1_classify", [str(i) for i in range(N_CHUNKS)], STATE, batch_size=1,
          params={"detector":"v1","chunk":CHUNK,"n_files":N_FILES}, verbose=False)
assert cls.n_done_batches()==N_CHUNKS, (
    f"nb1_classify is only {cls.n_done_batches()}/{N_CHUNKS} complete — "
    "this notebook needs NB1's classification to have finished")
HASH2T = {r["file_hash"]: set(r["t"]) for r in cls.rows()}
print(f"{N_FILES:,} workflow files · {len(HASH2T):,} carry at least one safeguard")
print("classes validated here:", ", ".join(TREATMENTS))

## 1 · Draw the sample and pull the file text

One streaming pass over `workflows.tar.gz`. Positive-stratum files are extracted wherever
they appear; everything else is subsampled deterministically on the file's own SHA-256, which
is uniform by construction, so the negative strata are a genuine random sample of the corpus
rather than a slice of whatever order the tar happens to be in.

The pass writes its result to Drive as soon as it finishes, and is skipped entirely on a
re-run.

In [ ]:
SAMPLE_PATH = Path(STATE,"nb7_sample.json")
rng = random.Random(SEED)

if SAMPLE_PATH.exists():
    S = json.loads(SAMPLE_PATH.read_text())
    print(f"sample already collected: {len(S['rows'])} files — skipping the tar pass")
else:
    # --- choose the positive stratum up front, from the stored classifications
    by_class = {t: [h for h,ts in HASH2T.items() if t in ts] for t in TREATMENTS}
    for t,v in by_class.items(): v.sort()          # deterministic before sampling
    want_pos = {}
    for t in TREATMENTS:
        n = min(N_POS_PER_CLASS, len(by_class[t]))
        for h in rng.sample(by_class[t], n): want_pos.setdefault(h, []).append(t)
    print("positive stratum:", {t:sum(1 for v in want_pos.values() if t in v) for t in TREATMENTS})

    near = {t: [] for t in TREATMENTS}   # reservoirs
    seen_near = {t: 0 for t in TREATMENTS}
    rand, seen_rand = [], 0
    got_pos = {}
    TARP = RAW/"workflows.tar.gz"
    t0, n = time.time(), 0

    def reservoir(pool, seen, k, item, r):
        if len(pool) < k: pool.append(item)
        else:
            j = r.randrange(seen)
            if j < k: pool[j] = item

    with tarfile.open(TARP, "r|gz") as tf:
        for m in tf:
            if not m.isfile(): continue
            h = m.name.rsplit("/",1)[-1]; n += 1
            is_pos = h in want_pos
            # uniform 1/SUBSAMPLE draw keyed on the content hash itself
            in_sub = (int(h[:6],16) % SUBSAMPLE)==0
            if not (is_pos or in_sub): continue
            try: text = tf.extractfile(m).read().decode("utf-8", errors="replace")
            except Exception: continue
            if is_pos:
                got_pos[h] = text
            if in_sub and h not in HASH2T:
                sig = prefilter(text)                 # keyword hit but classifier said no
                for t in sig:
                    seen_near[t]+=1
                    reservoir(near[t], seen_near[t], N_NEAR_PER_CLASS, (h,text), rng)
                if not sig:
                    seen_rand+=1
                    reservoir(rand, seen_rand, N_RANDOM, (h,text), rng)
            if n % 500_000 == 0:
                el=time.time()-t0
                print(f"  {n:,}/{N_FILES:,} files · {el/60:.1f} min · "
                      f"~{(N_FILES-n)/max(1,n/el)/60:.0f} min left", flush=True)

    rows=[]
    for h,ts in want_pos.items():
        if h in got_pos:
            for t in ts: rows.append({"hash":h,"cls":t,"stratum":"positive","text":got_pos[h]})
    for t in TREATMENTS:
        for h,txt in near[t]: rows.append({"hash":h,"cls":t,"stratum":"near_miss","text":txt})
    for h,txt in rand:
        rows.append({"hash":h,"cls":rng.choice(TREATMENTS),"stratum":"random","text":txt})
    S={"rows":rows,"n_scanned":n,"seen_near":seen_near,"seen_rand":seen_rand,"seed":SEED}
    atomic_write_text(SAMPLE_PATH, json.dumps(S))
    print(f"\nscanned {n:,} files in {(time.time()-t0)/60:.1f} min")

from collections import Counter
print("\nrows by stratum:", dict(Counter(r["stratum"] for r in S["rows"])))
print("rows by class  :", dict(Counter(r["cls"] for r in S["rows"])))
print("TOTAL          :", len(S["rows"]))

In [ ]:
# repository name for each sampled file — one pass over the history CSV
REPO_PATH = Path(STATE,"nb7_sample_repos.json")
if REPO_PATH.exists():
    H2R = json.loads(REPO_PATH.read_text())
else:
    want = {r["hash"] for r in S["rows"]}
    H2R = {}
    with gzip.open(RAW/"workflows.csv.gz","rt",errors="replace") as f:
        rd = csv.DictReader(f)
        hcol = next((c for c in rd.fieldnames if "hash" in c.lower()), None)
        rcol = next((c for c in rd.fieldnames if c.lower() in
                     ("repository","repo","repo_name","name")), None)
        pcol = next((c for c in rd.fieldnames if "path" in c.lower()), None)
        assert hcol and rcol, f"cannot find hash/repo columns in {rd.fieldnames}"
        for row in rd:
            h = row.get(hcol)
            if h in want and h not in H2R:
                H2R[h] = {"repo": row.get(rcol,""), "path": row.get(pcol,"") if pcol else ""}
                if len(H2R)==len(want): break
    atomic_write_text(REPO_PATH, json.dumps(H2R))
print(f"resolved {len(H2R)}/{len({r['hash'] for r in S['rows']})} files to a repository")

## 2 · Write the workbook

Two files come out of this cell. `detector_validation.xlsx` is what you label — it contains
the file, the question, and nothing else. `nb7_validation_key.json` holds what the classifier
said, and stays closed until scoring.

In [ ]:
QUESTION = {
 "T1_test":      "Does this workflow run an automated TEST SUITE as a check on pull requests?",
 "T2_lint":      "Does this workflow run a LINTER or static style check on pull requests?",
 "T3_coverage":  "Does this workflow report or enforce CODE COVERAGE on pull requests?",
 "T4_types":     "Does this workflow run a TYPE CHECKER on pull requests?",
 "T6_mergequeue":"Is this workflow triggered by a MERGE QUEUE (merge_group)?",
}
RULE = ("Answer YES only if BOTH hold: (a) the workflow actually runs that kind of check, and "
        "(b) it is triggered by pull_request or pull_request_target — except T6_mergequeue, "
        "which is judged on the merge_group trigger alone. A workflow that runs tests only on "
        "push, on a schedule, or on release is a NO. A housekeeping workflow (stale-bot, "
        "labeler, greetings) is a NO.")

import openpyxl
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter

rows = list(S["rows"])
random.Random(SEED+1).shuffle(rows)          # interleave strata to avoid order effects
for i,r in enumerate(rows,1): r["row_id"]=i

wb = openpyxl.Workbook()
ARIAL   = "Arial"
HEAD    = Font(name=ARIAL, bold=True, size=11, color="FFFFFF")
HFILL   = PatternFill("solid", fgColor="0F6B63")
INPUT   = PatternFill("solid", fgColor="FFFF00")
EXFILL  = PatternFill("solid", fgColor="EDF0EE")
THIN    = Border(bottom=Side(style="thin", color="CCCCCC"))

# ---------------- Instructions -------------------------------------------
ws = wb.active; ws.title = "Instructions"
ws.column_dimensions["A"].width = 108
def line(txt, bold=False, size=11, gap_after=0):
    c = ws.cell(row=line.r, column=1, value=txt)
    c.font = Font(name=ARIAL, bold=bold, size=size)
    c.alignment = Alignment(wrap_text=True, vertical="top")
    line.r += 1 + gap_after
line.r = 1
line("Detector validation — hand labelling", bold=True, size=16, gap_after=1)
line("Go to the 'Label' sheet and fill in the yellow LABEL column for every row. "
     "Row 2 is a filled-in EXAMPLE; leave it as it is.", gap_after=1)
line("The rule", bold=True, size=12)
line(RULE, gap_after=1)
line("Allowed answers", bold=True, size=12)
line("yes      — the workflow does this, and gates pull requests")
line("no       — it does not, or it runs on some other trigger")
line("unclear  — you genuinely cannot tell from the file (use sparingly; these are "
     "reported separately and excluded from precision and recall)", gap_after=1)
line("What each column is", bold=True, size=12)
line("row_id      identifier — do not change")
line("safeguard   which of the five detectors this row is about")
line("question    the exact question to answer for this file")
line("repository  where the file came from, for context")
line("workflow    the complete YAML, truncated at %d characters" % MAX_CHARS)
line("LABEL       yellow — your answer goes here")
line("notes       optional; useful when you answer 'unclear'", gap_after=1)
line("You are labelling blind on purpose", bold=True, size=12)
line("The classifier's own verdicts are NOT in this workbook. They are held in "
     "nb7_validation_key.json and joined to your labels only at scoring time, so that "
     "knowing the machine's answer cannot influence yours. Do not open that file first.",
     gap_after=1)
line("When you are done", bold=True, size=12)
line("Save the workbook back to the peltzman/results folder on Drive under the same name, "
     "then run cell 8 of NB7 to compute precision, recall and the confidence intervals.")

# ---------------- Label ---------------------------------------------------
ls = wb.create_sheet("Label")
cols = [("row_id",8),("safeguard",15),("question",46),("repository",30),
        ("workflow",95),("LABEL",11),("notes",28)]
for j,(name,w) in enumerate(cols,1):
    c = ls.cell(row=1, column=j, value=name); c.font=HEAD; c.fill=HFILL
    c.alignment = Alignment(horizontal="center", vertical="center")
    ls.column_dimensions[get_column_letter(j)].width = w
ls.freeze_panes = "A2"
ls.row_dimensions[1].height = 22

EXAMPLE = ("name: CI\non:\n  pull_request:\n    branches: [main]\njobs:\n  test:\n"
           "    runs-on: ubuntu-latest\n    steps:\n      - uses: actions/checkout@v4\n"
           "      - run: pip install -e .\n      - run: pytest -q")
body = Font(name=ARIAL, size=10)
mono = Font(name="Consolas", size=9)

def write_row(i, row_id, cls, repo, text, label="", note="", example=False):
    vals = [row_id, cls, QUESTION[cls], repo, text, label, note]
    for j,v in enumerate(vals,1):
        c = ls.cell(row=i, column=j, value=v)
        c.font = mono if j==5 else body
        c.alignment = Alignment(wrap_text=True, vertical="top")
        c.border = THIN
        if example: c.fill = EXFILL
        elif j==6:  c.fill = INPUT
    ls.row_dimensions[i].height = 150

write_row(2, "EXAMPLE", "T1_test", "octocat/example", EXAMPLE,
          label="yes", note="pytest, triggered by pull_request", example=True)
for i,r in enumerate(rows, start=3):
    meta = H2R.get(r["hash"], {})
    repo = meta.get("repo","(unknown)") + ("\n"+meta["path"] if meta.get("path") else "")
    txt  = r["text"][:MAX_CHARS] + ("\n\n… truncated …" if len(r["text"])>MAX_CHARS else "")
    write_row(i, r["row_id"], r["cls"], repo, txt)

dv = DataValidation(type="list", formula1='"yes,no,unclear"', allow_blank=True,
                    showDropDown=False, errorTitle="Not a valid label",
                    error="Use yes, no or unclear.")
ls.add_data_validation(dv); dv.add(f"F3:F{2+len(rows)}")

path = OUT/"detector_validation.xlsx"
wb.save(path)

key = {str(r["row_id"]): {"hash":r["hash"], "cls":r["cls"], "stratum":r["stratum"],
                          "predicted": int(r["cls"] in HASH2T.get(r["hash"], set()))}
       for r in rows}
atomic_write_text(Path(STATE,"nb7_validation_key.json"), json.dumps(key))
print(f"wrote {path}  ({len(rows)} rows to label)")
print(f"key held separately at {Path(STATE,'nb7_validation_key.json')}")
print("\nDownload the workbook, label column F, and save it back to the same path.")

## 3 · Score

Run this only after the workbook is labelled and saved back to
`peltzman/results/detector_validation.xlsx`.

Precision comes from the predicted-positive stratum. Recall is estimated on the near-miss
stratum and is deliberately conservative: because near-misses are where false negatives
concentrate, the recall reported here is a **lower bound** on recall over the corpus as a
whole, and the paper should say so.

In [ ]:
import pandas as pd
from math import sqrt
lab = pd.read_excel(OUT/"detector_validation.xlsx", sheet_name="Label")
lab = lab[lab.row_id.astype(str).str.isdigit()].copy()
lab["row_id"] = lab.row_id.astype(int)
key = json.loads(Path(STATE,"nb7_validation_key.json").read_text())
lab["cls"]       = lab.row_id.map(lambda i: key[str(i)]["cls"])
lab["stratum"]   = lab.row_id.map(lambda i: key[str(i)]["stratum"])
lab["predicted"] = lab.row_id.map(lambda i: key[str(i)]["predicted"])
lab["LABEL"]     = lab.LABEL.astype(str).str.strip().str.lower()

blank = int((~lab.LABEL.isin(["yes","no","unclear"])).sum())
assert blank == 0, f"{blank} rows are unlabelled or hold an unrecognised value"
unclear = lab[lab.LABEL=="unclear"]
sc = lab[lab.LABEL.isin(["yes","no"])].copy()
sc["truth"] = (sc.LABEL=="yes").astype(int)
print(f"{len(lab)} labelled · {len(unclear)} unclear (excluded) · {len(sc)} scored\n")

def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"),)*3
    p = k/n; d = 1+z*z/n; c = p + z*z/(2*n)
    h = z*sqrt(p*(1-p)/n + z*z/(4*n*n))
    return p, (c-h)/d, (c+h)/d

print(f"{'class':16s} {'precision':>22s} {'recall (lower bd)':>26s}")
out={}
for c in sorted(sc.cls.unique()):
    pos = sc[(sc.cls==c)&(sc.stratum=="positive")]
    nm  = sc[(sc.cls==c)&(sc.stratum=="near_miss")]
    tp, npos = int(pos.truth.sum()), len(pos)
    fn, nnm  = int(nm.truth.sum()), len(nm)          # near-miss judged yes = a miss
    p,pl,ph = wilson(tp,npos)
    # recall is only estimable where the near-miss stratum actually has rows;
    # with none sampled there is no evidence, and reporting 1.000 would be a lie
    r,rl,rh = wilson(tp, tp+fn) if nnm > 0 else (float("nan"),)*3
    out[c]={"tp":tp,"n_pos":npos,"precision":p,"prec_ci":[pl,ph],
            "fn_in_nearmiss":fn,"n_nearmiss":nnm,"recall_lb":r,"recall_ci":[rl,rh]}
    rec = (f"{r:6.3f} [{rl:.3f},{rh:.3f}]  {fn} miss / {nnm}" if nnm > 0
           else "   n/a  no near-misses sampled")
    print(f"{c:16s} {p:6.3f} [{pl:.3f},{ph:.3f}] n={npos:<3d}  {rec}")

rnd = sc[sc.stratum=="random"]
fp_rnd = int(rnd.truth.sum())
print(f"\nuniform-random negatives judged YES: {fp_rnd} of {len(rnd)} "
      f"— a non-zero count here means misses are not confined to near-misses")

agree = float((sc.truth == sc.predicted).mean())
print(f"raw agreement between labeller and classifier: {agree:.3f} over {len(sc)} decisions")

res={"per_class":out,"n_labelled":int(len(lab)),"n_unclear":int(len(unclear)),
     "n_scored":int(len(sc)),"random_negatives_yes":fp_rnd,"n_random":int(len(rnd)),
     "raw_agreement":agree,
     "sample_design":{"n_pos_per_class":N_POS_PER_CLASS,"n_near_per_class":N_NEAR_PER_CLASS,
                      "n_random":N_RANDOM,"subsample":SUBSAMPLE,"seed":SEED}}
atomic_write_text(Path(STATE,"nb7_validation.json"), json.dumps(res,indent=1,default=float))
pd.DataFrame(out).T.to_csv(OUT/"detector_validation.csv")
print(f"\nwrote {OUT}/detector_validation.csv and state/nb7_validation.json")